In [ ]:
# Injected by the pipeline or CI. Defaults to dev so a manual run
# is never the strict one.
env = "dev"


In [ ]:
# Generated from dataops/01-monitoring.yaml -- do not edit by hand.
#
# Evaluates every declared expectation and SLA against what actually landed,
# then applies the enforcement mode for this environment.
#
# The rules live in ttfabric.monitoring; this notebook only supplies the tables
# and decides what to do with the verdicts.

from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from ttfabric.monitoring import (
    run_expectations, evaluate_slas, FAIL, ERROR, WARN,
)

# `env` comes from the parameters cell above, overridden at run time by the
# pipeline or by tools/run_monitor.py.
try:
    environment = str(env).strip() or "dev"      # noqa: F821  (notebook parameter)
except NameError:
    environment = "dev"

EXPECTATIONS = [{'table': 'bronze_kastle_pg_arrivals',
  'layer': 'bronze',
  'checks': [{'id': 'BR-ARR-001',
              'rule': 'schema_match',
              'severity': 'critical',
              'registry_ref': 'kastle_pg.arrivals'},
             {'id': 'BR-ARR-002',
              'rule': 'row_count_delta',
              'severity': 'warning',
              'max_increase_pct': 50,
              'max_decrease_pct': 20},
             {'id': 'BR-ARR-003',
              'rule': 'freshness',
              'severity': 'error',
              'column': '_ingested_at',
              'max_age_hours': 26}],
  'expected_columns': ['id',
                       'card_holder_key',
                       'card_holder',
                       'card_holder_inst_key',
                       'building_key',
                       'date_key',
                       'time_key',
                       'reader_key',
                       'user_type',
                       'hour',
                       'used_presence_this_day',
                       'is_enrolled_today',
                       'company',
                       'user_type_label']},
 {'table': 'fct_arrivals',
  'layer': 'gold',
  'checks': [{'id': 'GL-ARR-001',
              'rule': 'freshness',
              'severity': 'error',
              'column': '_built_at',
              'max_age_hours': 26,
              'note': 'Daily build plus tolerance, matching the bronze freshness '
                      'window it depends on.'},
             {'id': 'GL-ARR-002',
              'rule': 'row_count_delta',
              'severity': 'warning',
              'max_increase_pct': 50,
              'max_decrease_pct': 20,
              'note': 'Same bounds as bronze BR-ARR-002: gold is a full rebuild of the '
                      'same 2,000 rows, so the two should move together. A divergence '
                      'means rows are being lost between layers.'}],
  'source': 'silver.stg_arrivals'},
 {'table': 'dim_card_holder',
  'layer': 'gold',
  'checks': [{'id': 'GL-CDH-001',
              'rule': 'freshness',
              'severity': 'error',
              'column': '_built_at',
              'max_age_hours': 26,
              'note': 'Daily build plus tolerance.'}],
  'source': 'silver.stg_card_holder'},
 {'table': 'bronze_kastle_pg_card_holder',
  'layer': 'bronze',
  'checks': [{'id': 'BR-CDH-001',
              'rule': 'schema_match',
              'severity': 'critical',
              'registry_ref': 'kastle_pg.card_holder'},
             {'id': 'BR-CDH-002',
              'rule': 'freshness',
              'severity': 'error',
              'column': '_ingested_at',
              'max_age_hours': 26}],
  'expected_columns': ['card_holder_key',
                       'card_holder',
                       'is_active_in_last_30_days',
                       'is_active_in_last_60_days',
                       'is_active_in_last_90_days',
                       'is_active_in_last_365_days']},
 {'table': 'bronze_kastle_pg_calendar_date',
  'layer': 'bronze',
  'checks': [{'id': 'BR-CAL-001',
              'rule': 'schema_match',
              'severity': 'critical',
              'registry_ref': 'kastle_pg.calendar_date'},
             {'id': 'BR-CAL-002',
              'rule': 'row_count_delta',
              'severity': 'warning',
              'max_increase_pct': 400,
              'max_decrease_pct': 5}],
  'expected_columns': ['date_key',
                       'date',
                       'is_work_day',
                       'day_is_holiday',
                       'date_range',
                       'day']},
 {'table': 'stg_arrivals',
  'layer': 'silver',
  'checks': [{'id': 'SL-ARR-001',
              'rule': 'not_null',
              'column': 'card_holder',
              'severity': 'error',
              'warn_above': 0.001,
              'fail_above': 0.01,
              'measured_on': 'input',
              'note': 'Measured 0/2000 null. Zero today, so any null at all is new and '
                      'worth an error.'},
             {'id': 'SL-ARR-002',
              'rule': 'accepted_values',
              'column': 'user_type',
              'values': [1, 2],
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output',
              'note': 'Measured 1696 Personnel + 304 Visitor = 2000, no third value. '
                      'Zero tolerance is a statement: a third category means the '
                      'source added one nobody announced.'},
             {'id': 'SL-ARR-003',
              'rule': 'referential_integrity',
              'column': 'building_key',
              'references': 'stg_building.building_key',
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output',
              'note': 'Measured 0 orphans across all five fact-to-dimension paths. '
                      'Kept because this is a synthetic demo database and a real '
                      'Kastle feed will not be this clean.'},
             {'id': 'SL-ARR-004',
              'rule': 'referential_integrity',
              'column': 'date_key',
              'references': 'stg_calendar_date.date_key',
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output',
              'note': 'Measured 0 orphans. Facts span 2025-11-03..2026-02-12 and the '
                      'calendar covers it, but a fact outside the calendar resolves to '
                      'the unknown member rather than failing -- this is what reports '
                      'it.'}],
  'source': 'bronze_kastle_pg_arrivals',
  'expected_columns': ['arrival_id',
                       'card_holder_key',
                       'card_holder_guid',
                       'institution_key',
                       'building_key',
                       'date_key',
                       'time_key',
                       'reader_key',
                       'user_type',
                       'hour_of_day',
                       'used_presence_this_day',
                       'is_enrolled_today',
                       'company_name',
                       'user_type_label',
                       'arrival_date']},
 {'table': 'stg_card_holder',
  'layer': 'silver',
  'checks': [{'id': 'SL-CDH-001',
              'rule': 'not_null',
              'column': 'card_holder_guid',
              'severity': 'critical',
              'warn_above': 0.001,
              'fail_above': 0.01,
              'measured_on': 'output',
              'note': 'The natural key. A null one means deduplication has nothing to '
                      'key on.'},
             {'id': 'SL-CDH-002',
              'rule': 'accepted_values',
              'column': 'is_active_in_last_365_days',
              'values': [0, 1],
              'severity': 'warning',
              'fail_above': 0.0,
              'measured_on': 'output',
              'note': 'Source value was constant 1 for all 300 rows (CON-001). Silver '
                      'recomputes it from arrivals.'}],
  'source': 'bronze_kastle_pg_card_holder',
  'expected_columns': ['card_holder_key',
                       'card_holder_guid',
                       'is_active_in_last_30_days',
                       'is_active_in_last_60_days',
                       'is_active_in_last_90_days',
                       'is_active_in_last_365_days']},
 {'table': 'stg_user_security',
  'layer': 'silver',
  'checks': [{'id': 'SL-USR-001',
              'rule': 'not_null',
              'column': 'inst_ids',
              'severity': 'warning',
              'warn_above': 0.7,
              'fail_above': 0.85,
              'measured_on': 'input',
              'note': 'Measured 4/6 null = 66.7% (NUL-001). default_nulls resolves it '
                      'to an explicit empty scope.'}],
  'source': 'bronze_kastle_pg_user_security',
  'expected_columns': ['user_security_id',
                       'username',
                       'persona',
                       'building_keys_json',
                       'inst_ids_json',
                       'reader_type',
                       'own_institution_id',
                       'scope_description',
                       'created_at',
                       'updated_at']},
 {'table': 'stg_calendar_date',
  'layer': 'silver',
  'checks': [{'id': 'SL-CAL-001',
              'rule': 'not_null',
              'column': 'full_date',
              'severity': 'critical',
              'warn_above': 0.0,
              'fail_above': 0.0,
              'measured_on': 'output',
              'note': 'Measured 0/104 null. Zero tolerance because a null date in a '
                      'marked date table breaks time intelligence silently -- it '
                      'returns blank, not an error (CAL-001).'}],
  'source': 'bronze_kastle_pg_calendar_date',
  'expected_columns': ['date_key',
                       'calendar_date',
                       'is_work_day',
                       'day_is_holiday',
                       'day_name',
                       'date_range_source',
                       'date_range']}]

SLAS = [{'table': 'fct_arrivals',
  'freshness_hours': 26,
  'completeness_pct': 99.9,
  'availability_pct': 99.0},
 {'table': 'dim_card_holder',
  'freshness_hours': 26,
  'completeness_pct': 99.9,
  'availability_pct': 99.0}]

ENFORCEMENT = {'dev': {'mode': 'warn', 'block_on': []},
 'qa': {'mode': 'block', 'block_on': ['error', 'critical']},
 'uat': {'mode': 'block', 'block_on': ['error', 'critical']},
 'prod': {'mode': 'alert', 'block_on': []}}

# Where each table lives. The monitor spans three storage items of two
# different kinds; nothing else in the pipeline crosses all of them at once.
LOCATIONS = {'bronze_kastle_pg_arrivals': ('lakehouse', 'lh_bronze'),
 'fct_arrivals': ('warehouse', 'wh_gold'),
 'stg_arrivals': ('lakehouse', 'lh_silver'),
 'dim_card_holder': ('warehouse', 'wh_gold'),
 'stg_card_holder': ('lakehouse', 'lh_silver'),
 'bronze_kastle_pg_card_holder': ('lakehouse', 'lh_bronze'),
 'bronze_kastle_pg_calendar_date': ('lakehouse', 'lh_bronze'),
 'stg_user_security': ('lakehouse', 'lh_silver'),
 'bronze_kastle_pg_user_security': ('lakehouse', 'lh_bronze'),
 'stg_calendar_date': ('lakehouse', 'lh_silver'),
 'stg_building': ('lakehouse', 'lh_silver')}
DEFAULT_LAKEHOUSE = 'lh_silver'

RESULTS_TABLE = 'dq_run_log_monitor'
run_id = f"mon_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"

print(f"environment {environment}   run {run_id}")
print(f"{len(EXPECTATIONS)} tables, "
      f"{sum(len(e['checks']) for e in EXPECTATIONS)} checks, {len(SLAS)} SLAs")
print()


def read(name):
    """Resolve a table name to a DataFrame, wherever that layer actually lives.

    The monitor spans all three layers, and they are NOT in one place:

      bronze   lh_bronze    a different lakehouse -- must be qualified
      silver   lh_silver    this notebook's default -- unqualified works
      gold     wh_gold      a WAREHOUSE, which spark.read.table cannot reach
                            at all; it needs the Fabric DW connector

    Reading everything unqualified silently resolves against the default
    lakehouse, so every bronze and gold check fails as TABLE_OR_VIEW_NOT_FOUND
    -- which looks like missing data rather than a wrong lookup.
    """
    bare = name.rsplit(".", 1)[-1]
    kind, item = LOCATIONS.get(bare, ("lakehouse", None))

    if kind == "warehouse":
        # Lazy import: the connector registers spark.read.synapsesql as a side
        # effect, and is unavailable until it has been imported at least once.
        import com.microsoft.spark.fabric                  # noqa: F401
        from com.microsoft.spark.fabric.Constants import Constants  # noqa: F401
        schema = name.rsplit(".", 2)[-2] if "." in name else "dbo"
        return spark.read.synapsesql(f"{item}.{schema}.{bare}")   # noqa: F821

    if item and item != DEFAULT_LAKEHOUSE:
        return spark.read.table(f"{item}.{bare}")      # noqa: F821
    return spark.read.table(bare)                          # noqa: F821


# Previous row counts, so row_count_delta has something to compare against.
# Absent on a first run, which the rule reports as "not measurable" rather
# than as a breach.
previous_rows = {}
try:
    history = spark.read.table(RESULTS_TABLE)             # noqa: F821
    latest = history.filter(F.col("rule_id") == "row_count_delta") \
        .groupBy("table_name").agg(F.max("run_id").alias("run_id"))
    for row in history.join(latest, ["table_name", "run_id"]).collect():
        if row["rows"] is not None:
            previous_rows[row["table_name"]] = row["rows"]
except Exception as exc:
    print(f"no monitoring history yet ({type(exc).__name__}); "
          f"row_count_delta will be skipped on this run")

run = run_expectations(read, EXPECTATIONS, previous_rows=previous_rows)
run.results.extend(evaluate_slas(read, SLAS))

for result in run.results:
    print(result)

counts = run.counts()
print()
print("  ".join(f"{status}={count}" for status, count in sorted(counts.items())))

# --- persist -------------------------------------------------------------
rows = [(run_id, r.check_id, r.table, r.layer, r.rule, r.severity, r.status,
         float(r.measured) if r.measured is not None else None,
         float(r.limit) if r.limit is not None else None,
         int(r.rows) if r.rows is not None else None,
         r.detail, datetime.now(timezone.utc))
        for r in run.results]

schema = ("run_id string, check_id string, table_name string, layer string, "
          "rule_id string, severity string, status string, measured double, "
          "limit_value double, rows bigint, detail string, checked_at timestamp")
# mergeSchema: this is an append-only log, and adding a column to it should not
# be what stops monitoring from running. Delta otherwise rejects the whole write
# on any schema difference.
spark.createDataFrame(rows, schema).write.mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(RESULTS_TABLE)                            # noqa: F821
print(f"\n{len(rows)} results appended to {RESULTS_TABLE}")

# --- enforce -------------------------------------------------------------
policy = ENFORCEMENT.get(environment, {"mode": "warn", "block_on": []})
blocking = run.blocking(policy.get("block_on") or [])

print(f"\nenforcement for {environment}: mode={policy.get('mode')} "
      f"block_on={policy.get('block_on') or 'nothing'}")

if blocking:
    for result in blocking:
        print(f"  BLOCKING  {result}")
    # Raised so the pipeline stops. In an environment whose block_on is empty
    # this never fires, which is the point of declaring it per environment.
    raise RuntimeError(
        f"{len(blocking)} blocking breach(es) in {environment}: "
        + ", ".join(r.check_id for r in blocking))

breached = [r for r in run.results if r.status in (FAIL, ERROR, WARN)]
if breached:
    print(f"\n{len(breached)} breach(es) recorded, none blocking in {environment}.")
else:
    print("\nall checks passed")
